# CMGAN — Clone, Inference, và Finetune

Notebook này gồm 3 phần:
1. Clone repo CMGAN và cài dependencies
2. Inference (enhance) trên tập TEST bằng checkpoint pretrained có sẵn
3. Finetune tiếp checkpoint pretrained trên dữ liệu TRAIN của bạn (20 epochs)

Chỉnh các biến đường dẫn trong mỗi cell cho đúng với dataset của bạn trước khi chạy.

## 1. Clone repo & cài đặt

In [1]:
!git clone https://github.com/ruizhecao96/CMGAN.git /kaggle/working/CMGAN
!pip install -q pesq joblib natsort librosa

Cloning into '/kaggle/working/CMGAN'...
remote: Enumerating objects: 453, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 453 (delta 130), reused 97 (delta 97), pack-reused 297 (from 1)
Receiving objects: 100% (453/453), 92.35 MiB | 35.70 MiB/s, done.
Resolving deltas: 100% (179/179), done.
  Preparing metadata (setup.py) ... done


## 2. Inference (enhance) trên tập TEST

Dùng checkpoint pretrained có sẵn trong repo (`src/best_ckpt/ckpt`), không cần clean reference.

In [2]:
import os, glob, sys
import numpy as np
import soundfile as sf
import torch

# ==== Chỉnh các đường dẫn này ====
REPO_DIR = '/kaggle/working/CMGAN/src'
CKPT_PATH = '/kaggle/working/CMGAN/src/best_ckpt/ckpt'
TEST_INPUT_DIR = '/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TEST'
ENHANCED_OUTPUT_DIR = '/kaggle/working/enhanced_cmgan'
# =================================

sys.path.insert(0, REPO_DIR)
from utils import power_compress, power_uncompress
from models import generator as gen_module

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

def load_and_resample(path, target_sr=16000):
    wav, sr = sf.read(path, dtype='float32')
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != target_sr:
        try:
            import librosa
            wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
        except ImportError:
            from scipy.signal import resample_poly
            from math import gcd
            g = gcd(sr, target_sr)
            wav = resample_poly(wav, target_sr // g, sr // g).astype(np.float32)
    return wav.astype(np.float32)

n_fft, hop = 400, 100
window = torch.hamming_window(n_fft).to(DEVICE)

model = gen_module.TSCNet(num_channel=64, num_features=n_fft // 2 + 1)
state_dict = torch.load(CKPT_PATH, map_location='cpu')
if isinstance(state_dict, dict) and 'state_dict' in state_dict:
    state_dict = state_dict['state_dict']
model.load_state_dict(state_dict)
model.to(DEVICE)
model.eval()
print('Đã load checkpoint:', CKPT_PATH)

@torch.no_grad()
def enhance_one_track(audio_path, cut_len=16000 * 16):
    noisy = load_and_resample(audio_path, 16000)
    noisy = torch.from_numpy(noisy).unsqueeze(0).to(DEVICE)

    c = torch.sqrt(noisy.size(-1) / torch.sum((noisy ** 2.0), dim=-1))
    noisy = torch.transpose(noisy, 0, 1)
    noisy = torch.transpose(noisy * c, 0, 1)

    length = noisy.size(-1)
    frame_num = int(np.ceil(length / 100))
    padded_len = frame_num * 100
    padding_len = padded_len - length
    noisy = torch.cat([noisy, noisy[:, :padding_len]], dim=-1)
    if padded_len > cut_len:
        batch_size = int(np.ceil(padded_len / cut_len))
        while 100 % batch_size != 0:
            batch_size += 1
        noisy = torch.reshape(noisy, (batch_size, -1))

    noisy_spec = torch.stft(noisy, n_fft, hop, window=window, onesided=True, return_complex=False)
    noisy_spec = power_compress(noisy_spec).permute(0, 1, 3, 2)
    est_real, est_imag = model(noisy_spec)
    est_real, est_imag = est_real.permute(0, 1, 3, 2), est_imag.permute(0, 1, 3, 2)

    est_spec_uncompress = power_uncompress(est_real, est_imag).squeeze(1)
    est_audio = torch.istft(
        torch.view_as_complex(est_spec_uncompress.contiguous()),
        n_fft, hop, window=window, onesided=True,
    )
    est_audio = est_audio / c
    est_audio = torch.flatten(est_audio)[:length].cpu().numpy()
    return est_audio

os.makedirs(ENHANCED_OUTPUT_DIR, exist_ok=True)
wav_files = sorted(glob.glob(os.path.join(TEST_INPUT_DIR, '**', '*.wav'), recursive=True))
print(f'Tìm thấy {len(wav_files)} file trong TEST.')

for i, wav_path in enumerate(wav_files, 1):
    rel_name = os.path.basename(wav_path)
    out_path = os.path.join(ENHANCED_OUTPUT_DIR, rel_name)
    try:
        est_audio = enhance_one_track(wav_path)
        sf.write(out_path, est_audio, 16000)
        print(f'[{i}/{len(wav_files)}] OK: {rel_name}')
    except Exception as e:
        print(f'[{i}/{len(wav_files)}] LỖI với {rel_name}: {e}')

print('Xong. File enhanced nằm ở:', ENHANCED_OUTPUT_DIR)

Device: cuda
Đã load checkpoint: /kaggle/working/CMGAN/src/best_ckpt/ckpt
Tìm thấy 0 file trong TEST.
Xong. File enhanced nằm ở: /kaggle/working/enhanced_cmgan


### Dọn bộ nhớ GPU trước khi sang phần Finetune

**Bắt buộc chạy cell này** trước khi sang phần 3, nếu không GPU sẽ bị đầy VRAM
(model của phần Inference vẫn còn giữ trong bộ nhớ) và phần Finetune sẽ báo lỗi
`CUDA out of memory`.

In [3]:
import gc
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print('VRAM đã dùng:', torch.cuda.memory_allocated() / 1024**3, 'GiB')
    print('VRAM reserved:', torch.cuda.memory_reserved() / 1024**3, 'GiB')
print('Đã dọn bộ nhớ xong.')

VRAM đã dùng: 1.9073486328125e-06 GiB
VRAM reserved: 0.001953125 GiB
Đã dọn bộ nhớ xong.


### Nén kết quả enhanced thành zip (tùy chọn)

In [4]:
import shutil
shutil.make_archive('/kaggle/working/enhanced_cmgan', 'zip', ENHANCED_OUTPUT_DIR)
print('Đã nén xong: /kaggle/working/enhanced_cmgan.zip')

Đã nén xong: /kaggle/working/enhanced_cmgan.zip


## 3. Finetune từ checkpoint pretrained

Yêu cầu: file trong thư mục NOISE và CLEAN phải **trùng tên nhau**.
Script tự tách 5% dữ liệu làm validation, lưu 1 checkpoint sau mỗi epoch.

**Lưu ý VRAM:** `BATCH_SIZE=2` và `CUT_LEN=16000` (1 giây/clip) là cấu hình an toàn cho GPU T4 (~14.5GB).
Nếu vẫn bị `CUDA out of memory`, giảm tiếp `BATCH_SIZE` xuống 1.
Nếu chạy mượt và muốn train nhanh/chất lượng hơn, có thể tăng dần `CUT_LEN` (vd 32000) hoặc `BATCH_SIZE`
và theo dõi log VRAM để canh chỉnh.

In [5]:
import os, sys, random, logging, gc, re, glob, shutil
import numpy as np
import soundfile as sf
import torch
import torch.nn.functional as F
from joblib import Parallel, delayed

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')

# ==== Chinh cac duong dan / hyperparams nay ====
REPO_DIR = '/kaggle/working/CMGAN/src'
INIT_CKPT = '/kaggle/working/CMGAN/src/best_ckpt/ckpt'   # checkpoint pretrained goc (dung neu chua finetune lan nao)
NOISY_DIR = '/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/NOISE'
CLEAN_DIR = '/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/CLEAN'
FINETUNE_OUTPUT_DIR = '/kaggle/working/finetuned_cmgan'

# Thu muc chua cac checkpoint da finetune tu lan chay truoc (dataset da upload len Kaggle, read-only)
PREV_FINETUNED_DIR = '/kaggle/input/datasets/foxduck/cmgan-ft/finetuned_cmgan'

TOTAL_EPOCHS = 20   # tong so epoch muon dat duoc (khong phai so epoch train them)
BATCH_SIZE = 2
LR = 1e-4
CUT_LEN = 16000
VAL_RATIO = 0.05
LOG_INTERVAL = 100
NUM_WORKERS = 2
# ================================================

sys.path.insert(0, REPO_DIR)
from models.generator import TSCNet
from models.discriminator import Discriminator
from utils import power_compress, power_uncompress

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
logging.info(f'Dung device: {DEVICE}')

os.makedirs(FINETUNE_OUTPUT_DIR, exist_ok=True)

# ==== Tim checkpoint da finetune gan nhat (epoch cao nhat) tu lan chay truoc ====
def find_latest_checkpoint(ckpt_dir):
    pattern = re.compile(r'epoch(\d+)_valloss([\d.]+)\.pt$')
    best = None
    if not os.path.isdir(ckpt_dir):
        return None
    for p in glob.glob(os.path.join(ckpt_dir, '*.pt')):
        m = pattern.search(os.path.basename(p))
        if m:
            epoch = int(m.group(1))
            if best is None or epoch > best[0]:
                best = (epoch, p)
    return best  # (epoch, path) hoac None

latest = find_latest_checkpoint(PREV_FINETUNED_DIR)
if latest:
    START_EPOCH, RESUME_CKPT = latest
    logging.info(f'Tim thay checkpoint da finetune o epoch {START_EPOCH}: {RESUME_CKPT}')
    # copy toan bo checkpoint cu vao thu muc output de sau nay zip du bo (khong bat buoc)
    for p in glob.glob(os.path.join(PREV_FINETUNED_DIR, '*.pt')):
        dst = os.path.join(FINETUNE_OUTPUT_DIR, os.path.basename(p))
        if not os.path.isfile(dst):
            shutil.copy(p, dst)
else:
    START_EPOCH, RESUME_CKPT = 0, INIT_CKPT
    logging.info('Khong tim thay checkpoint finetune truoc do trong PREV_FINETUNED_DIR, '
                 'se train tu INIT_CKPT (checkpoint pretrained goc).')

REMAINING_EPOCHS = TOTAL_EPOCHS - START_EPOCH
assert REMAINING_EPOCHS > 0, (
    f'Da hoan thanh du {TOTAL_EPOCHS} epoch roi (checkpoint moi nhat la epoch {START_EPOCH}), '
    f'khong can train them. Neu muon train them, tang TOTAL_EPOCHS len.'
)
logging.info(f'Se train tiep {REMAINING_EPOCHS} epoch nua: tu epoch {START_EPOCH + 1} den epoch {TOTAL_EPOCHS}.')

def load_and_resample(path, target_sr=16000):
    wav, sr = sf.read(path, dtype='float32')
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != target_sr:
        try:
            import librosa
            wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
        except ImportError:
            from scipy.signal import resample_poly
            from math import gcd
            g = gcd(sr, target_sr)
            wav = resample_poly(wav, target_sr // g, sr // g).astype(np.float32)
    return wav.astype(np.float32)

class PairedDataset(torch.utils.data.Dataset):
    def __init__(self, noisy_dir, clean_dir, file_names, cut_len):
        self.noisy_dir = noisy_dir
        self.clean_dir = clean_dir
        self.file_names = file_names
        self.cut_len = cut_len

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        name = self.file_names[idx]
        noisy = load_and_resample(os.path.join(self.noisy_dir, name))
        clean = load_and_resample(os.path.join(self.clean_dir, name))
        length = min(len(noisy), len(clean))
        noisy, clean = noisy[:length], clean[:length]

        if length < self.cut_len:
            units = self.cut_len // length
            noisy = np.concatenate([noisy] * units + [noisy[: self.cut_len % length]])
            clean = np.concatenate([clean] * units + [clean[: self.cut_len % length]])
        else:
            start = random.randint(0, length - self.cut_len)
            noisy = noisy[start:start + self.cut_len]
            clean = clean[start:start + self.cut_len]

        return torch.from_numpy(clean), torch.from_numpy(noisy)

def pesq_loss(clean, noisy, sr=16000):
    try:
        from pesq import pesq
        return pesq(sr, clean, noisy, 'wb')
    except Exception:
        return -1

def batch_pesq(clean, noisy, device):
    scores = Parallel(n_jobs=-1)(delayed(pesq_loss)(c, n) for c, n in zip(clean, noisy))
    scores = np.array(scores)
    if -1 in scores:
        return None
    scores = (scores - 1) / 3.5
    return torch.FloatTensor(scores).to(device)

def forward_generator_step(model, n_fft, hop, window, clean, noisy, device):
    c = torch.sqrt(noisy.size(-1) / torch.sum((noisy ** 2.0), dim=-1))
    noisy_t, clean_t = torch.transpose(noisy, 0, 1), torch.transpose(clean, 0, 1)
    noisy_t, clean_t = torch.transpose(noisy_t * c, 0, 1), torch.transpose(clean_t * c, 0, 1)

    noisy_spec = torch.stft(noisy_t, n_fft, hop, window=window, onesided=True, return_complex=False)
    clean_spec = torch.stft(clean_t, n_fft, hop, window=window, onesided=True, return_complex=False)

    noisy_spec = power_compress(noisy_spec).permute(0, 1, 3, 2)
    clean_spec = power_compress(clean_spec)
    clean_real = clean_spec[:, 0, :, :].unsqueeze(1)
    clean_imag = clean_spec[:, 1, :, :].unsqueeze(1)

    est_real, est_imag = model(noisy_spec)
    est_real, est_imag = est_real.permute(0, 1, 3, 2), est_imag.permute(0, 1, 3, 2)
    est_mag = torch.sqrt(est_real ** 2 + est_imag ** 2)
    clean_mag = torch.sqrt(clean_real ** 2 + clean_imag ** 2)

    est_spec_uncompress = power_uncompress(est_real, est_imag).squeeze(1)
    est_audio = torch.istft(
        torch.view_as_complex(est_spec_uncompress.contiguous()),
        n_fft, hop, window=window, onesided=True,
    )

    return {
        'est_real': est_real, 'est_imag': est_imag, 'est_mag': est_mag,
        'clean_real': clean_real, 'clean_imag': clean_imag, 'clean_mag': clean_mag,
        'est_audio': est_audio, 'clean': clean_t,
    }

def generator_loss(discriminator, out, one_labels, loss_weights):
    predict_fake = discriminator(out['clean_mag'], out['est_mag'])
    gan_loss = F.mse_loss(predict_fake.flatten(), one_labels)
    mag_loss = F.mse_loss(out['est_mag'], out['clean_mag'])
    ri_loss = F.mse_loss(out['est_real'], out['clean_real']) + F.mse_loss(out['est_imag'], out['clean_imag'])
    length = out['est_audio'].size(-1)
    time_loss = torch.mean(torch.abs(out['est_audio'] - out['clean'][:, :length]))
    return (loss_weights[0] * ri_loss + loss_weights[1] * mag_loss
            + loss_weights[2] * time_loss + loss_weights[3] * gan_loss)

def discriminator_loss(discriminator, out, one_labels, device):
    length = out['est_audio'].size(-1)
    est_list = list(out['est_audio'].detach().cpu().numpy())
    clean_list = list(out['clean'].cpu().numpy()[:, :length])
    scores = batch_pesq(clean_list, est_list, device)
    if scores is None:
        return None
    fake = discriminator(out['clean_mag'], out['est_mag'].detach())
    real = discriminator(out['clean_mag'], out['clean_mag'])
    return F.mse_loss(real.flatten(), one_labels) + F.mse_loss(fake.flatten(), scores)

noisy_files = set(os.listdir(NOISY_DIR))
clean_files = set(os.listdir(CLEAN_DIR))
common = sorted(noisy_files & clean_files)
if not common:
    raise RuntimeError('Khong tim thay file trung ten nao giua NOISY_DIR va CLEAN_DIR.')
logging.info(f'Tim thay {len(common)} cap file khop ten.')

random.seed(42)
random.shuffle(common)
n_val = max(1, int(len(common) * VAL_RATIO))
val_files, train_files = common[:n_val], common[n_val:]
logging.info(f'Train: {len(train_files)} file, Val: {len(val_files)} file')

train_ds = PairedDataset(NOISY_DIR, CLEAN_DIR, train_files, CUT_LEN)
val_ds = PairedDataset(NOISY_DIR, CLEAN_DIR, val_files, CUT_LEN)
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS
)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

n_fft, hop = 400, 100
window = torch.hamming_window(n_fft).to(DEVICE)

model = TSCNet(num_channel=64, num_features=n_fft // 2 + 1).to(DEVICE)
state = torch.load(RESUME_CKPT, map_location='cpu')
if isinstance(state, dict) and 'state_dict' in state:
    state = state['state_dict']
model.load_state_dict(state)
logging.info(f'Da load checkpoint: {RESUME_CKPT}')

discriminator = Discriminator(ndf=16).to(DEVICE)
loss_weights = [0.1, 0.9, 0.2, 0.05]

if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    used_mem = torch.cuda.memory_allocated() / 1024**3
    logging.info(f'GPU tong: {total_mem:.2f} GiB, dang dung sau khi load model: {used_mem:.2f} GiB')

opt_g = torch.optim.AdamW(model.parameters(), lr=LR)
opt_d = torch.optim.AdamW(discriminator.parameters(), lr=2 * LR)
sched_g = torch.optim.lr_scheduler.StepLR(opt_g, step_size=max(1, TOTAL_EPOCHS // 3), gamma=0.5)
sched_d = torch.optim.lr_scheduler.StepLR(opt_d, step_size=max(1, TOTAL_EPOCHS // 3), gamma=0.5)

# Optimizer/scheduler cua lan chay truoc khong duoc luu lai (chi luu model.state_dict()),
# nen "tua" scheduler toi dung vi tri LR ma no le ra da o epoch START_EPOCH truoc khi train tiep.
for _ in range(START_EPOCH):
    sched_g.step()
    sched_d.step()
logging.info(f'LR hien tai sau khi tua scheduler: g={opt_g.param_groups[0]["lr"]:.2e}, '
             f'd={opt_d.param_groups[0]["lr"]:.2e}')

for epoch in range(START_EPOCH, TOTAL_EPOCHS):
    model.train()
    discriminator.train()
    for step, (clean, noisy) in enumerate(train_loader, 1):
        clean, noisy = clean.to(DEVICE), noisy.to(DEVICE)
        one_labels = torch.ones(clean.size(0)).to(DEVICE)

        try:
            out = forward_generator_step(model, n_fft, hop, window, clean, noisy, DEVICE)
            g_loss = generator_loss(discriminator, out, one_labels, loss_weights)
            opt_g.zero_grad()
            g_loss.backward()
            opt_g.step()

            d_loss = discriminator_loss(discriminator, out, one_labels, DEVICE)
            if d_loss is not None:
                opt_d.zero_grad()
                d_loss.backward()
                opt_d.step()
                d_loss_val = d_loss.item()
            else:
                d_loss_val = 0.0
        except torch.cuda.OutOfMemoryError:
            logging.info(f'Bo qua step {step} do het bo nho GPU (file co the qua dai).')
            opt_g.zero_grad()
            opt_d.zero_grad()
            gc.collect()
            torch.cuda.empty_cache()
            continue

        if step % LOG_INTERVAL == 0:
            logging.info(f'Epoch {epoch+1}/{TOTAL_EPOCHS} Step {step}/{len(train_loader)} '
                         f'g_loss={g_loss.item():.4f} d_loss={d_loss_val:.4f}')

    model.eval()
    discriminator.eval()
    val_g_loss_total = 0.0
    with torch.no_grad():
        for clean, noisy in val_loader:
            clean, noisy = clean.to(DEVICE), noisy.to(DEVICE)
            one_labels = torch.ones(clean.size(0)).to(DEVICE)
            out = forward_generator_step(model, n_fft, hop, window, clean, noisy, DEVICE)
            val_g_loss_total += generator_loss(discriminator, out, one_labels, loss_weights).item()
    val_g_loss = val_g_loss_total / max(1, len(val_loader))
    logging.info(f'== Epoch {epoch+1} xong. Val generator loss: {val_g_loss:.4f} ==')

    ckpt_path = os.path.join(FINETUNE_OUTPUT_DIR, f'cmgan_finetuned_epoch{epoch+1}_valloss{val_g_loss:.4f}.pt')
    torch.save(model.state_dict(), ckpt_path)
    logging.info(f'Da luu checkpoint: {ckpt_path}')

    sched_g.step()
    sched_d.step()

logging.info(f'Hoan tat finetune. Da train den epoch {TOTAL_EPOCHS}. Cac checkpoint nam o: {FINETUNE_OUTPUT_DIR}')

2026-09-14 02:00:11,159 Dung device: cuda
2026-09-14 02:00:11,174 Tim thay checkpoint da finetune o epoch 17: /kaggle/input/datasets/foxduck/cmgan-ft/finetuned_cmgan/cmgan_finetuned_epoch17_valloss0.0604.pt
2026-09-14 02:00:13,710 Se train tiep 3 epoch nua: tu epoch 18 den epoch 20.
2026-09-14 02:00:13,986 Tim thay 8460 cap file khop ten.
2026-09-14 02:00:13,990 Train: 8037 file, Val: 423 file
2026-09-14 02:00:14,101 Da load checkpoint: /kaggle/input/datasets/foxduck/cmgan-ft/finetuned_cmgan/cmgan_finetuned_epoch17_valloss0.0604.pt
2026-09-14 02:00:14,126 GPU tong: 14.56 GiB, dang dung sau khi load model: 0.01 GiB
/tmp/ipykernel_23/2333469802.py:233: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/do

### Nén các checkpoint đã finetune thành zip (tùy chọn)

In [6]:
import shutil
shutil.make_archive('/kaggle/working/finetuned_cmgan', 'zip', FINETUNE_OUTPUT_DIR)
print('Đã nén xong: /kaggle/working/finetuned_cmgan.zip')

Đã nén xong: /kaggle/working/finetuned_cmgan.zip
